In [1]:
import argparse
import os

import anndata as ad

from methyltrain.api.prepare import prepare_cohort
from methyltrain.config.loader import load_config
from methyltrain.fs.layout import CohortLayout
from methyltrain.api.steps import save_cohort

from typing import Dict, Tuple

import anndata as ad
import pandas as pd

from methyltrain.fs.layout import ProjectLayout, CohortLayout
from methyltrain.api.steps import (
    download, 
    clean_data, 
    quality_control, 
    preprocess, 
    aggregate_cohort, 
    filter_by_mad,
    cohort_batch_correction,
    aggregate_genes,
    clip_and_scale,
    split,
    load_raw_project,
    load_processed_project,
    save_cohort
)

verbose = True
config = "/ddn_exa/campbell/sli/methyltrain/config/pancancer_config.yaml"

In [2]:
# Load the user-provided configurations
config = load_config(config)
cohort = config.get('project_id', '')

# Initialize the projects provided in the configurations
project_dir = config.get('project_dir', '')
all_projects = config.get('projects', [])
project_list = []
for project in all_projects:
    project_list.append(os.path.join(project_dir, f"{project}_adata.h5ad"))

# Initialize the default cohort layout
layout = CohortLayout(
    cohort_name = cohort,
    root_dir = "./data",
    project_list = project_list,
    cohort_adata = f"../data/cohort/{cohort}_cohort_adata.h5ad",
    train_adata = f"../data/cohort/{cohort}_train_adata.h5ad",
    val_adata = f"../data/cohort/{cohort}_val_adata.h5ad",
    test_adata = f"../data/cohort/{cohort}_test_adata.h5ad"
)
layout.initialize()
layout.validate()

In [7]:
# Load each processed project AnnData object 
if verbose: print("Attempting to load all project AnnData objects")
project_adatas = [load_processed_project(path) 
                  for path in layout.project_list]
if verbose: print("Successfully loaded all project AnnData objects")

# Aggregate the projects into a cohort AnnData object
if verbose: print("Attempting to aggregate the cohort")
cohort_adata = aggregate_cohort(project_adatas, layout)
if verbose: print("Successfully aggregated the cohort")

# Perform MAD probe filtering to reduce probe dimensionality
if config.get('toggles', {}).get('MAD_probe_filtering', True):
    if verbose: print("Attempting to perform MAD probe filtering")
    cohort_adata = filter_by_mad(cohort_adata, config)
    if verbose: print("Successfully performed MAD probe filtering")

# Perform batch effect correction across datasets if toggled
if config.get('toggles', {}).get('batch_correction', True):
    if verbose: print("Attempting to perform batch correction")
    cohort_adata = cohort_batch_correction(cohort_adata, config)
    if verbose: print("Successfully performed batch correction")

# Optionally aggregate to the gene-level if toggled by user configurations
if config.get('toggles', {}).get('gene_aggregation', True):
    if verbose: print("Attempting to perform gene aggregation")
    cohort_adata = aggregate_genes(cohort_adata, config)
    if verbose: print("Successfully aggregated to the gene-level")

# Winsorize and scale M-values to [-1, 1] promote downstream ML stability
if config.get('toggles', {}).get('clip_and_scale', True):
    if verbose: print("Attempting to perform winsorization and Min-max scaling")
    cohort_adata = clip_and_scale(cohort_adata, config)
    if verbose: print("Successfully winsorized and scaled the cohort")

# Split the cohort AnnData object into train-val-test splits
if config.get('toggles', {}).get('split', True):
    if verbose: print("Attempting to split the cohort into training sets")
    train_adata, val_adata, test_adata = split(cohort_adata, config)
    if verbose: print("Successfully split the cohort into training sets")
else:
    if verbose: print("Did not split the cohort into training sets (not toggled)")
    train_adata, val_adata, test_adata = None, None, None

# Save the processed cohort adata prior to splitting
if verbose: print("Attempting to save the cohort AnnData object")
save_cohort(cohort_adata, layout)
if verbose: print("Successfully saved the cohort AnnData object")

Attempting to load all project AnnData objects
Successfully loaded all project AnnData objects
Attempting to aggregate the cohort
Successfully aggregated the cohort
Did not split the cohort into training sets (not toggled)
Attempting to save the cohort AnnData object
Successfully saved the cohort AnnData object


In [8]:
# Save the training splits if toggled
if config.get('toggles', {}).get('split', True):
    train_adata.write_h5ad(layout.train_adata)
    val_adata.write_h5ad(layout.val_adata)
    test_adata.write_h5ad(layout.test_adata)

In [4]:
adata = ad.read_h5ad("/ddn_exa/campbell/sli/methylcdm-project/data/training/methylation/pancancer_test_adata.h5ad")
adata

AnnData object with n_obs × n_vars = 1074 × 169747
    obs: 'file_name', 'data_type', 'data_category', 'experimental_strategy', 'platform', 'project_id', 'submitter_id', 'sample_type', 'aliquot_id', 'status', 'barcode', 'missing_rate', 'mean'
    uns: 'cohort_id', 'conversion', 'data_type', 'level', 'platform', 'preprocessing_steps', 'projects', 'reference_genome', 'split', 'split_percentage', 'state'